# Adding LC Data to a Dataframe of EBs

In [ ]:
import lightkurve as lk
from lightkurve import search_lightcurve
import matplotlib.pyplot as plt
import requests
import pandas as pd
from time import perf_counter
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

%matplotlib inline


---

In [ ]:
def queryMAST(KOI):
    """
    Use the MAST API to get info about the canonical name and KIC of a Kepler Object of Interest (KOI).
    Returns: JSON object with information about object with it's KIC as one of it's fields.
    """
    # Resolve KOI -> Kepler ID / canonical name using Exo.MAST
    info = requests.get(
        "https://exo.mast.stsci.edu/api/v0.1/exoplanets/identifiers/",
        params={"name": KOI},
        timeout=30,
    ).json()
    
    return info


In [ ]:
def fetchStitchedLC(KIC):
    """
    Given a KIC, fetch it's light curve data using LC.
    Returns: Stitched LC object.
    """
    search = lk.search_lightcurve(
        f"KIC {KIC}",
        mission="Kepler",
        author="Kepler",
        cadence="long"
    )

    return search.download_all().stitch().remove_nans()


In [ ]:
def getLCArraysOriginal(kic):
    lc = fetchStitchedLC(kic)
    
    return pd.Series({
        "time": lc.time.value,
        "flux": lc.flux.value,
        "flux_err": lc.flux_err.value if lc.flux_err is not None else None,
        "n_points": len(lc),
    })


In [ ]:
def getLCArrays(kic):
    """
    Fetches LC arrays and returns a pandas Series. 
    Includes error handling to prevent pipeline crashes.
    """
    try:
        lc = fetchStitchedLC(kic)
        
        if lc is None:
            return pd.Series({"time": None, "flux": None, "flux_err": None, "n_points": 0}, name=kic)
            
        return pd.Series({
            "time": lc.time.value,
            "flux": lc.flux.value,
            "flux_err": lc.flux_err.value if lc.flux_err is not None else None,
            "n_points": len(lc),
        }, name=kic)
        
    except Exception as e:
        # Catch network timeouts or corrupted downloads and return empty/NaN data
        # so the rest of the dataset can continue processing.
        return pd.Series({"time": None, "flux": None, "flux_err": None, "n_points": 0}, name=kic)


In [ ]:
def fetchLightCurvesParallel(df, max_workers=10):
    """
    Parallelizes the downloading and feature extraction of lightcurves.
    """
    kic_list = df.index.tolist()
    results = []

    # Using ThreadPoolExecutor because network requests are I/O bound
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Map KICs to their future objects
        future_to_kic = {executor.submit(getLCArrays, kic): kic for kic in kic_list}

        # as_completed yields futures as they finish, allowing the progress bar to update
        for future in tqdm(as_completed(future_to_kic), total=len(kic_list), desc="Downloading Kepler LCs"):
            results.append(future.result())

    # Compile all processed series into a new DataFrame
    features_df = pd.DataFrame(results)
    
    # Join the new features back to the original DataFrame based on the KIC index
    return df.join(features_df)


In [ ]:
def plotLightCurve(df, kic, with_error=True):
    """
    Plot the light curve for a given KIC from a DataFrame.
    """
    row = df[df["KIC"] == kic]
    
    if row.empty:
        raise ValueError(f"KIC {kic} not found in DataFrame.")
    
    row = row.iloc[0]
    
    t = row["time"]
    f = row["flux"]
    e = row.get("flux_err", None)

    # Plot
    plt.figure(figsize=(20, 5))
    
    if with_error and e is not None:
        plt.errorbar(t, f, yerr=e, fmt='-', linewidth=1)
    else:
        plt.plot(t, f, linewidth=1)
    
    plt.xlabel("Time")
    plt.ylabel(r"Normalized Flux (e$^{-}$ s$^{-1}$)")
    plt.title(f"KIC {kic} Light Curve")
    plt.show()
    

In [ ]:
def sampleRandomKEB(df):
    idx = np.random.randint(0, len(df))
    kic = int(df.loc[idx]['KIC'])

    return kic


---

In [ ]:
df = pd.read_csv(r"../assets/data/kepler-eclipsing-binary-catalog.csv")


In [ ]:
df = df.drop(columns=["Unnamed: 11"])


In [ ]:
df.head()


In [ ]:
# listOfKICs = df['KIC'].astype(int).tolist()


---

Now, using a copy of the dataframe to test converting the lightcurve into arrays from being an LC object.

In [ ]:
copyDf = df.head(20).copy()


In [ ]:
# copyDf[["time", "flux", "flux_err", "n_points"]] = copyDf["KIC"].apply(getLCArrays)


In [ ]:
copyDf


In [ ]:
plotLightCurve(copyDf, 3863594)


In [ ]:
# Doesn't really make sense 'cos the LC seems to be normalized already
# df["flux_mean"] = df["flux"].apply(np.mean)
# df["flux_std"] = df["flux"].apply(np.std)


In [ ]:
copyDf


---

In [ ]:
# df[["time", "flux", "flux_err", "n_points"]] = df["KIC"].apply(getLCArrays)


> **This took 212 mins to get halfway through the dataset.**

---

In [ ]:
kebLCs = fetchLightCurvesParallel(copyDf, max_workers=6)


In [ ]:
kebLCs.head()


In [ ]:
# df.to_json("../assets/data/keb-lcs.json")
